In [ ]:
import os
import glob
import zipfile
import re
import numpy as np

# re = string and text manipulation

# glob = pattern matching tool, using like *txt

# Structure: UT/raw (for zips) and UT/ifgramStack (for extracted matrices)
PROJECT_NAME = "UT"

workspace_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
project_dir = os.path.join(workspace_dir, "data", PROJECT_NAME)

zip_dir = os.path.join(project_dir, "raw")
out_dir = os.path.join(project_dir, "inputs", "ifgramStack")

print(f"[*] Scanning {zip_dir} for HyP3 zip files...")
os.makedirs(out_dir, exist_ok=True)

zip_files = glob.glob(os.path.join(zip_dir, "*.zip"))
total_zips = len(zip_files)

if total_zips == 0:
    print(f"[!] FATAL: No .zip files found in {zip_dir}.")
    print(f"    Make sure your downloaded files are mapped to data/{PROJECT_NAME}/raw/")
else:
    print(f"[*] FOUND {total_zips} ZIP FILES.")
    print(f"[*] Extracting structurally to: {out_dir} \n")

    # Iterate through each zip
    for i, zf in enumerate(zip_files, 1):
        basename = os.path.basename(zf)

        # Parse Dates from Filename (HyP3 format: S1AA_20250805T123456_20250910T123456...)
        matches = re.findall(r'(\d{8})T\d{6}', basename)
        if len(matches) < 2:
            print(f"[{i}/{total_zips}] [!] Missing valid dates in filename: {basename}. Skipping.")
            continue

        ref_date, sec_date = matches[0], matches[1]
        pair_folder_name = f"{ref_date}_{sec_date}"

        # Create the specific pair directory MintPy needs
        pair_dir = os.path.join(out_dir, pair_folder_name)
        os.makedirs(pair_dir, exist_ok=True)

        print(f"[{i}/{total_zips}] Processing Pair: {pair_folder_name}")

        try:
            with zipfile.ZipFile(zf, 'r') as z:
                namelist = z.namelist()

                # Define exactly what 4 geometric/physics files we need + the config text file!
                required_targets = {
                    'Phase': ['unw_phase.tif'],
                    'Coherence': ['corr.tif', 'coh.tif'],
                    'DEM': ['dem.tif'],
                    'Incidence angle': ['lv_theta.tif', 'inc_map.tif'],
                    'Metadata': ['.txt']
                }

                # Extract only the targeted files into the pair directory
                for tag, suffixes in required_targets.items():
                    # For metadata, ensure we don't accidentally match another txt file by making sure it's the main file
                    if tag == 'Metadata':
                        matched_files = [f for f in namelist if f.endswith('.txt') and 'README' not in f and 'parameters' not in f]
                    else:
                        matched_files = [f for f in namelist if any(f.endswith(s) for s in suffixes)]

                    if matched_files:
                        target_file_in_zip = matched_files[0]
                        out_file_name = os.path.basename(target_file_in_zip)
                        out_file_path = os.path.join(pair_dir, out_file_name)

                        if not os.path.exists(out_file_path):
                            print(f"    -> Extracting {tag:15} | {out_file_name}")
                            with z.open(target_file_in_zip) as source, open(out_file_path, "wb") as target:
                                target.write(source.read())
                        else:
                            pass
                    else:
                        print(f"    [!] Missing {tag} file in {basename}")

        except zipfile.BadZipFile:
            print(f"    [!] FATAL: {basename} is corrupted. Please re-download.")

    print(f"\n[+] STACK ARCHITECTURE COMPLETE.")
    print(f"[*] Ready for MintPy config. View tree in: {out_dir}")

## Time-Series Setup (MintPy Template)
The extracted TIFFs are useless to MintPy until we write the **Configuration Template**. This file acts as the steering wheel for the MintPy engine, telling it exactly where the data is, what algorithms to use for atmospheric correction, and what reference point to anchor the entire time-series to.


cd F:\S1CL\M9_insar\data\UT; conda run --no-capture-output -n insar_env python -m mintpy.cli.smallbaselineApp mintpy_config.txt --start modify_network


In [ ]:
import os
import glob
from osgeo import gdal

# GDAL is the C-based workhorse for geospatial manipulation. We use it to align matrices.
gdal.UseExceptions()

PROJECT_NAME = "UT"
workspace_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
out_dir = os.path.join(workspace_dir, "data", PROJECT_NAME, "inputs", "ifgramStack")

print("[*] Calculating Absolute Common Intersection for all HyP3 matrices...")
unw_files = glob.glob(os.path.join(out_dir, "*", "*unw_phase.tif"))

min_x_list, max_y_list, max_x_list, min_y_list = [], [], [], []

for f in unw_files:
    ds = gdal.Open(f)
    gt = ds.GetGeoTransform()
    cols = ds.RasterXSize
    rows = ds.RasterYSize

    # Calculate Bounding Box coordinates
    ulx = gt[0]
    uly = gt[3]
    lrx = ulx + gt[1] * cols
    lry = uly + gt[5] * rows # gt[5] is negative pixel height

    min_x_list.append(ulx)
    max_y_list.append(uly)
    max_x_list.append(lrx)
    min_y_list.append(lry)
    ds = None # Release memory

# Mathematical intersection (Tightest possible bound that fits ALL images)
inter_ulx = max(min_x_list)
inter_uly = min(max_y_list) # ULY is highest Y. Minimum of maximums gives intersection.
inter_lrx = min(max_x_list)
inter_lry = max(min_y_list) # LRY is lowest Y. Maximum of minimums gives intersection.

print(f"[+] Common Bounds: UL=({inter_ulx:.4f}, {inter_uly:.4f}) | LR=({inter_lrx:.4f}, {inter_lry:.4f})")

# RUN THIS IN A NEW CELL TO DIAGNOSE
print(f"Calculated ULX: {inter_ulx}, LRX: {inter_lrx}")
print(f"Calculated ULY: {inter_uly}, LRY: {inter_lry}")

if inter_ulx >= inter_lrx or inter_lry >= inter_uly:
    print("[!] FATAL: No common intersection found! The stack contains files that do not overlap.")
    # Check if you have files from different regions or misplaced granules
else:
    print("[+] Intersection is valid. Checking for disk/path issues.")


In [ ]:
# ----------------------------------------------------------------------
# SELECTIVE STACK ALIGNMENT: Deduplicating Frames for a Clean Stack
# ----------------------------------------------------------------------
import numpy as np
from osgeo import gdal
import os
import glob
import shutil

# This logic automatically picks one consistent frame per date-pair folder.
# It resolves the "operands could not be broadcast" error caused by duplicate frames.

aligned_dir = os.path.join(project_dir, "inputs", "aligned_ifgramStack")

# Clean start for the aligned stack
if os.path.exists(aligned_dir):
    shutil.rmtree(aligned_dir)
os.makedirs(aligned_dir, exist_ok=True)

# Find all date-pair folders
pair_folders = glob.glob(os.path.join(out_dir, "20*"))
print(f"[*] Found {len(pair_folders)} date-pair folders in raw extraction.")

# Setup Global Boundary Reference (from Cell 3)
# We need pixel resolution. We'll peek at the first file found.
first_unw_all = glob.glob(os.path.join(out_dir, "20*", "*unw_phase.tif"))[0]
ds_ref = gdal.Open(first_unw_all)
gt_ref = ds_ref.GetGeoTransform()
proj_ref = ds_ref.GetProjection()
pixel_w, pixel_h = gt_ref[1], gt_ref[5]
ds_ref = None

cols = int(round((inter_lrx - inter_ulx) / abs(pixel_w)))
rows = int(round((inter_uly - inter_lry) / abs(pixel_h)))

print(f"[*] Target Alignment Grid: {cols}x{rows} pixels at {abs(pixel_w):.6f} resolution.")

deduped_count = 0
for folder in pair_folders:
    # Find all unw files in this folder
    unw_files = glob.glob(os.path.join(folder, "*unw_phase.tif"))
    if not unw_files:
        continue

    # PICK ONLY THE FIRST ONE (The deduplication event)
    target_unw = sorted(unw_files)[0]

    # Get the prefix (everything before _unw_phase.tif)
    # We use _unw_phase.tif as the split point to cleanly extract the unique ID prefix
    if "_unw_phase.tif" in target_unw:
        prefix = target_unw.split("_unw_phase.tif")[0]
        unw_suffix = "_unw_phase.tif"
    else:
        # Fallback for alternative naming
        prefix = target_unw.replace(".unw_phase.tif", "")
        unw_suffix = ".unw_phase.tif"

    # Define the 5 critical files for this specific prefix
    files_to_process = [
        prefix + unw_suffix,
        prefix + "_corr.tif",
        prefix + "_dem.tif",
        prefix + "_inc_map.tif",
        prefix + ".txt"
    ]

    # Create target directory
    pair_name = os.path.basename(folder)
    target_pair_dir = os.path.join(aligned_dir, pair_name)
    os.makedirs(target_pair_dir, exist_ok=True)

    # Process each file
    for fpath in files_to_process:
        if not os.path.exists(fpath):
            # Try alternative coherence name if _corr.tif is missing
            if "_corr.tif" in fpath:
                alt = fpath.replace("_corr.tif", "_coh.tif")
                if os.path.exists(alt): fpath = alt
            # Try alternative incidence name
            elif "_inc_map.tif" in fpath:
                alt = fpath.replace("_inc_map.tif", "_lv_theta.tif")
                if os.path.exists(alt): fpath = alt
            else:
                # If it's a critical file (Phase/TXT), and still missing, skip pair
                if fpath.endswith(".txt") or fpath.endswith(unw_suffix):
                    print(f"    [!] Critical missing: {os.path.basename(fpath)}")
                    continue
                continue

        filename = os.path.basename(fpath)
        out_path = os.path.join(target_pair_dir, filename)

        if fpath.endswith(".tif"):
            gdal.Warp(out_path, fpath, format="GTiff",
                      outputBounds=(inter_ulx, inter_lry, inter_lrx, inter_uly),
                      width=cols, height=rows, dstSRS=proj_ref,
                      resampleAlg=gdal.GRA_NearestNeighbour, dstNodata=0)
        else:
            shutil.copy2(fpath, out_path)

    deduped_count += 1

print(f"\n[+] DEDUPLICATED ALIGNMENT COMPLETE.")
print(f"[*] Processed {deduped_count} unique date pairs.")
print(f"[*] Aligned Stack: {aligned_dir}")

In [ ]:
# ----------------------------------------------------------------------
# TRIGGER MINTPY ENGINE: load_data and modify_network
# ----------------------------------------------------------------------
import os

# move into the UT data directory so relative paths in the config work.
os.chdir(project_dir)

# CLEANUP: Delete stale metadata and HDF5 files.
# This is THE fix for the "broadcasting" error. MintPy was reading a
# stale 'coherenceSpatialAvg.txt' with 60 entries instead of 83.
print("[*] Performing Forensic Cleanup of stale MintPy artifacts...")
stale_files = [
    "inputs/ifgramStack.h5",
    "inputs/geometryGeo.h5",
    "coherenceSpatialAvg.txt" # <--- The "Ghost" that was causing the crash
]

for f in stale_files:
    if os.path.exists(f):
        os.remove(f)
        print(f"    -> Purged: {f}")

# --start load_data: Converts aligned TIFFs/Metadata into HDF5 objects
# --stop modify_network: Generates the coherence network plot for visual audit
print("\n[*] Running smallbaselineApp.py...")
!smallbaselineApp.py inputs/mintpy_config.txt --start load_data --stop modify_network

print("\n[+] DATA LOADED AND NETWORK GENERATED.")
print("[*] Inspect the 'network.pdf' or 'coherenceMatrix.pdf' in your UT/ directory to check stack health.")

In [ ]:
import os

# --- 1. RE-ESTABLISH DIRECTORY PATHS ---
current_cwd = os.getcwd()

if current_cwd.endswith("UT"):
    project_dir = current_cwd
elif "data" in os.listdir(current_cwd):
    project_dir = os.path.join(current_cwd, "data", "UT")
elif "UT" in os.listdir(current_cwd):
    project_dir = os.path.join(current_cwd, "UT")
else:
    while "data" not in os.listdir(os.getcwd()) and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
    project_dir = os.path.join(os.getcwd(), "data", "UT")

os.chdir(project_dir)
print(f"[*] Secure Working Directory: {os.getcwd()}")

# --- 2. EXECUTE THE PIPELINE FROM THE REFERENCE POINT ---
if os.path.exists("inputs/mintpy_config.txt"):
    print("[+] Config verified. Launching automatic pipeline from step: reference_point...")
    print("[*] Sit back. This will sequentially build your geocoded velocity maps.")

    # Run from reference_point all the way to geocode automatically
    !smallbaselineApp.py inputs/mintpy_config.txt --start reference_point

    print("\n[+] SUCCESS! The entire InSAR stack is inverted, corrected, and geocoded.")
else:
    print("[!] FATAL: inputs/mintpy_config.txt not found.")